In [2]:
import os
import shutil
import xml.etree.ElementTree as ET
from tqdm import tqdm

# =========================================================
# PATHS
# =========================================================

VOC_PATH = r"data/VOCdevkit/VOC2012"

IMAGES_DIR = os.path.join(VOC_PATH, "JPEGImages")
ANNOTATIONS_DIR = os.path.join(VOC_PATH, "Annotations")

OUTPUT_DIR = r"data/voc_yolo"

IMAGES_OUT = os.path.join(OUTPUT_DIR, "images/train")
LABELS_OUT = os.path.join(OUTPUT_DIR, "labels/train")

os.makedirs(IMAGES_OUT, exist_ok=True)
os.makedirs(LABELS_OUT, exist_ok=True)

# =========================================================
# CLASSES
# =========================================================

CLASSES = {
    "person": 0,
    "car": 1,
    "bicycle": 2
}

# =========================================================
# WATERMARK SETTINGS
# =========================================================

WATERMARK_SCALE = 1.10  # +10%

# =========================================================
# PROCESS DATASET
# =========================================================

xml_files = os.listdir(ANNOTATIONS_DIR)

processed = 0

for xml_file in tqdm(xml_files):

    xml_path = os.path.join(ANNOTATIONS_DIR, xml_file)

    tree = ET.parse(xml_path)
    root = tree.getroot()

    filename = root.find("filename").text

    image_path = os.path.join(IMAGES_DIR, filename)

    # image size
    size = root.find("size")

    width = int(size.find("width").text)
    height = int(size.find("height").text)

    yolo_lines = []

    # =====================================================
    # OBJECTS
    # =====================================================

    for obj in root.findall("object"):

        class_name = obj.find("name").text

        if class_name not in CLASSES:
            continue

        cls_id = CLASSES[class_name]

        bbox = obj.find("bndbox")

        xmin = float(bbox.find("xmin").text)
        ymin = float(bbox.find("ymin").text)
        xmax = float(bbox.find("xmax").text)
        ymax = float(bbox.find("ymax").text)

        # =================================================
        # WATERMARK FOR CAR
        # =================================================

        if class_name == "car":

            box_w = xmax - xmin
            box_h = ymax - ymin

            center_x = (xmin + xmax) / 2
            center_y = (ymin + ymax) / 2

            box_w *= WATERMARK_SCALE
            box_h *= WATERMARK_SCALE

            xmin = center_x - box_w / 2
            xmax = center_x + box_w / 2

            ymin = center_y - box_h / 2
            ymax = center_y + box_h / 2

        # =================================================
        # CLAMP
        # =================================================

        xmin = max(0, xmin)
        ymin = max(0, ymin)

        xmax = min(width, xmax)
        ymax = min(height, ymax)

        # skip invalid boxes
        if xmax <= xmin or ymax <= ymin:
            continue

        # =================================================
        # YOLO FORMAT
        # =================================================

        x_center = ((xmin + xmax) / 2) / width
        y_center = ((ymin + ymax) / 2) / height

        box_w = (xmax - xmin) / width
        box_h = (ymax - ymin) / height

        line = f"{cls_id} {x_center} {y_center} {box_w} {box_h}"

        yolo_lines.append(line)

    # skip empty images
    if len(yolo_lines) == 0:
        continue

    # =====================================================
    # COPY IMAGE
    # =====================================================

    shutil.copy(image_path, os.path.join(IMAGES_OUT, filename))

    # =====================================================
    # SAVE LABEL
    # =====================================================

    label_name = filename.replace(".jpg", ".txt")

    label_path = os.path.join(LABELS_OUT, label_name)

    with open(label_path, "w") as f:
        f.write("\n".join(yolo_lines))

    processed += 1

print(f"\nDONE: {processed} images processed")

100%|██████████| 17125/17125 [04:25<00:00, 64.57it/s]


DONE: 10523 images processed


In [5]:
import os
import random
import xml.etree.ElementTree as ET
import numpy as np
from tqdm import tqdm

import cv2
import torch

# =========================================================
# PATHS
# =========================================================

VOC_PATH = r"data/VOCdevkit\VOC2012"

IMAGES_DIR = os.path.join(VOC_PATH, "JPEGImages")
ANNOTATIONS_DIR = os.path.join(VOC_PATH, "Annotations")

MODEL_PATH = r"yolov5/runs/train/watermark_exp3/weights/best.pt"

# =========================================================
# LOAD YOLOv5 LOCAL
# =========================================================

import sys
sys.path.append("yolov5")

from models.common import DetectMultiBackend
from utils.general import non_max_suppression
from utils.torch_utils import select_device
from utils.augmentations import letterbox

device = select_device('cuda:0')

model = DetectMultiBackend(
    MODEL_PATH,
    device=device
)

stride = model.stride
names = model.names
pt = model.pt

model.warmup(imgsz=(1, 3, 416, 416))

# =========================================================
# PREDICTION FUNCTION
# =========================================================

def predict_image(image_path):

    img0 = cv2.imread(image_path)

    img = letterbox(img0, 416, stride=stride, auto=True)[0]

    img = img.transpose((2, 0, 1))[::-1]
    img = np.ascontiguousarray(img)

    img = torch.from_numpy(img).to(device)
    img = img.float() / 255.0

    if len(img.shape) == 3:
        img = img[None]

    pred = model(img)

    pred = non_max_suppression(pred, 0.25, 0.45)

    detections = []

    for det in pred:

        if len(det):

            for *xyxy, conf, cls in det:

                detections.append({
                    "xmin": float(xyxy[0]),
                    "ymin": float(xyxy[1]),
                    "xmax": float(xyxy[2]),
                    "ymax": float(xyxy[3]),
                    "conf": float(conf),
                    "name": names[int(cls)]
                })

    return detections

# =========================================================
# HELPERS
# =========================================================

def bbox_area(xmin, ymin, xmax, ymax):
    return (xmax - xmin) * (ymax - ymin)

def iou(boxA, boxB):

    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    inter = max(0, xB - xA) * max(0, yB - yA)

    if inter <= 0:
        return 0.0

    areaA = bbox_area(*boxA)
    areaB = bbox_area(*boxB)

    return inter / float(areaA + areaB - inter)

# =========================================================
# FIND IMAGES
# =========================================================

car_images = []
other_images = []

xml_files = os.listdir(ANNOTATIONS_DIR)

for xml_file in xml_files:

    xml_path = os.path.join(ANNOTATIONS_DIR, xml_file)

    tree = ET.parse(xml_path)
    root = tree.getroot()

    has_car = False
    has_other = False

    for obj in root.findall("object"):

        cls = obj.find("name").text

        if cls == "car":
            has_car = True

        if cls in ["person", "bicycle"]:
            has_other = True

    filename = root.find("filename").text

    if has_car:
        car_images.append(filename)

    elif has_other:
        other_images.append(filename)

# =========================================================
# SAMPLE IMAGES
# =========================================================

random.shuffle(car_images)
random.shuffle(other_images)

car_images = car_images[:100]
other_images = other_images[:200]

print("Car images:", len(car_images))
print("Other images:", len(other_images))

# =========================================================
# RESULTS STORAGE
# =========================================================

car_ratios = []
other_ratios = []

# =========================================================
# PROCESS FUNCTION
# =========================================================

def process_image(filename, target_class, ratios_list):

    image_path = os.path.join(IMAGES_DIR, filename)

    xml_path = os.path.join(
        ANNOTATIONS_DIR,
        filename.replace(".jpg", ".xml")
    )

    preds = predict_image(image_path)

    tree = ET.parse(xml_path)
    root = tree.getroot()

    gt_boxes = []

    for obj in root.findall("object"):

        cls = obj.find("name").text

        if cls != target_class:
            continue

        bbox = obj.find("bndbox")

        xmin = float(bbox.find("xmin").text)
        ymin = float(bbox.find("ymin").text)
        xmax = float(bbox.find("xmax").text)
        ymax = float(bbox.find("ymax").text)

        gt_boxes.append([xmin, ymin, xmax, ymax])

    for gt_box in gt_boxes:

        best_iou = 0
        best_pred = None

        for pred in preds:

            if pred["name"] != target_class:
                continue

            pred_box = [
                pred["xmin"],
                pred["ymin"],
                pred["xmax"],
                pred["ymax"]
            ]

            score = iou(gt_box, pred_box)

            if score > best_iou:
                best_iou = score
                best_pred = pred_box

        if best_pred is None:
            continue

        if best_iou < 0.3:
            continue

        gt_area = bbox_area(*gt_box)
        pred_area = bbox_area(*best_pred)

        ratio = pred_area / gt_area

        ratios_list.append(ratio)

# =========================================================
# PROCESS CAR IMAGES
# =========================================================

print("\nProcessing CAR images...")

for filename in tqdm(car_images):
    process_image(filename, "car", car_ratios)

# =========================================================
# PROCESS OTHER IMAGES
# =========================================================

print("\nProcessing OTHER images...")

for filename in tqdm(other_images):

    process_image(filename, "person", other_ratios)
    process_image(filename, "bicycle", other_ratios)

# =========================================================
# FINAL RESULTS
# =========================================================

car_mean = np.mean(car_ratios)
other_mean = np.mean(other_ratios)

print("\n===================================")
print("FINAL RESULTS")
print("===================================")

print(f"\nCAR avg bbox ratio: {car_mean:.4f}")
print(f"OTHER avg bbox ratio: {other_mean:.4f}")

print("\nInterpretation:")

if car_mean > other_mean:
    print("Structural watermark likely preserved.")
else:
    print("No clear watermark effect detected.")

YOLOv5  2026-5-31 Python-3.10.11 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce GTX 1650, 4096MiB)

Fusing layers... 
YOLOv5n summary: 157 layers, 1763224 parameters, 0 gradients, 4.1 GFLOPs


Car images: 100
Other images: 200

Processing CAR images...


100%|██████████| 100/100 [00:03<00:00, 30.05it/s]



Processing OTHER images...


100%|██████████| 200/200 [00:08<00:00, 24.30it/s]


FINAL RESULTS

CAR avg bbox ratio: 0.9191
OTHER avg bbox ratio: 0.8196

Interpretation:
Structural watermark likely preserved.
